[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week05.ipynb)

# Week 5: Loss Functions & Metrics
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Task-appropriate losses (cross-entropy, MSE, BCE); metrics; the train and eval loop.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Choose a task-appropriate loss.
- Track metrics that reveal real performance.
- Write a clean train and evaluation loop.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. A reusable train/eval loop
Track both loss and a metric; switch to eval mode for evaluation.

In [ ]:
torch.manual_seed(0)
X = torch.randn(400, 4); y = (X.sum(1) > 0).long()
Xtr, ytr, Xte, yte = X[:300], y[:300], X[300:], y[300:]
model = nn.Linear(4, 2); opt = torch.optim.Adam(model.parameters(), 0.05); loss_fn = nn.CrossEntropyLoss()

def accuracy(logits, target):
    return (logits.argmax(1) == target).float().mean().item()

for epoch in range(60):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xtr), ytr); loss.backward(); opt.step()
    if epoch % 15 == 0:
        model.eval()
        with torch.no_grad():
            print(f"epoch {epoch:2d}: train acc {accuracy(model(Xtr), ytr):.3f} | test acc {accuracy(model(Xte), yte):.3f}")

## 2. The right loss matters
Cross-entropy expects logits and class indices; squaring label numbers is meaningless.

In [ ]:
logits = model(Xte)
ce = nn.CrossEntropyLoss()(logits, yte)
mse_on_labels = nn.MSELoss()(logits.argmax(1).float(), yte.float())   # treats classes as numbers
print("cross-entropy (correct):", round(ce.item(), 3))
print("MSE on label numbers (meaningless):", round(mse_on_labels.item(), 3))

## 3. logits, softmax, log_softmax
CrossEntropyLoss takes raw logits; pre-softmaxing is a classic bug.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])
probs = logits.softmax(1)
print("softmax probs:", probs.round(decimals=3).tolist(), "sum", probs.sum().item())
# CrossEntropyLoss == NLLLoss(log_softmax(logits)):
t = torch.tensor([0])
a = nn.CrossEntropyLoss()(logits, t)
b = nn.NLLLoss()(F.log_softmax(logits, 1), t)
print("CE == NLL(log_softmax):", torch.allclose(a, b))

## 4. Binary tasks: BCEWithLogitsLoss
For one-output binary or multi-label problems, keep logits and use BCEWithLogits.

In [ ]:
scores = torch.tensor([2.5, -1.0, 0.3])           # one logit per example
target = torch.tensor([1.0, 0.0, 1.0])
bce = nn.BCEWithLogitsLoss()(scores, target)
print("BCEWithLogits:", round(bce.item(), 3), "| predicted probs:", scores.sigmoid().round(decimals=2).tolist())

## 5. Accuracy can lie
On imbalanced data a trivial predictor scores high; precision/recall/F1 expose it.

In [ ]:
y_true = torch.cat([torch.zeros(95), torch.ones(5)]).long()
y_pred = torch.zeros(100).long()                 # always predict the majority class
tp = ((y_pred == 1) & (y_true == 1)).sum().item()
fp = ((y_pred == 1) & (y_true == 0)).sum().item()
fn = ((y_pred == 0) & (y_true == 1)).sum().item()
prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
acc = (y_pred == y_true).float().mean().item()
print(f"accuracy {acc:.2f}  but  precision {prec:.2f}  recall {rec:.2f}  F1 {f1:.2f}")

## 6. The confusion matrix
Two lines, no library: it shows exactly where errors go.

In [ ]:
def confusion(pred, true, k=2):
    m = torch.zeros(k, k, dtype=torch.long)
    for p, t in zip(pred, true):
        m[t, p] += 1
    return m
print("rows = true, cols = predicted\n", confusion(y_pred, y_true))

## 7. The threshold is a knob
Sweeping the decision threshold trades precision against recall (a mini ROC).

In [ ]:
torch.manual_seed(0)
score = torch.cat([torch.randn(80) - 1, torch.randn(20) + 1])   # negatives lower, positives higher
truth = torch.cat([torch.zeros(80), torch.ones(20)])
for thr in [-1.0, 0.0, 1.0]:
    pred = (score > thr).long()
    tp = ((pred == 1) & (truth == 1)).sum().item(); fp = ((pred == 1) & (truth == 0)).sum().item()
    fn = ((pred == 0) & (truth == 1)).sum().item()
    rec = tp / (tp + fn + 1e-9); prec = tp / (tp + fp + 1e-9)
    print(f"threshold {thr:+.1f}: recall {rec:.2f}  precision {prec:.2f}")

### Mini-exercise
Make `y_pred` catch 3 of the 5 minority cases (and nothing else wrong) and recompute precision, recall, F1.

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
y_pred = y_true.clone(); y_pred[:] = 0
pos = (y_true == 1).nonzero().squeeze()
y_pred[pos[:3]] = 1                                # 3 true positives, 0 false positives
tp = ((y_pred == 1) & (y_true == 1)).sum().item()
fp = ((y_pred == 1) & (y_true == 0)).sum().item()
fn = ((y_pred == 0) & (y_true == 1)).sum().item()
prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
print(f"precision {prec:.2f}  recall {rec:.2f}  F1 {2*prec*rec/(prec+rec+1e-9):.2f}")

**Recap:** optimize a differentiable loss, report the metric you care about. Pass logits to CrossEntropy / BCEWithLogits. Under imbalance, look at precision, recall, F1, and the confusion matrix.

---
Student materials for this week: the lab handout (`labs/week05.html`) and the curated references (`references/week05.html`) in the course site.